In [2]:
"""
build_golden_set.py

Builds a STRATIFIED DRAFT of the golden evaluation set for the Hiver
SDE assignment (SpotifyCares brand). This does NOT produce final gold
labels -- it pulls a representative, difficulty-varied sample and pre-fills
everything that can be derived mechanically (thread transcript, a
suggested/regex intent as a labeling aid, an auto difficulty flag) so a
human only has to fill in the judgment columns.

Sampling strategy:
    - 10 examples from each of the 17 named intent buckets (rule-classified)
    - 20 examples from the Other/Uncategorised bucket (this is the taxonomy's
      long tail and needs real representation, not just the clean cases)
    - Within every stratum, mix short/simple and longer/multi-turn threads
      rather than sampling purely at random, so difficulty varies
    - Auto-flags a message as "likely hard" if it matches 2+ intent regexes
      (genuine ambiguity) or is very short (<6 words, likely low-signal)
      -- human labelers should double check these first

Output: golden_eval_set_draft.csv with columns:
    example_id, conversation_id, tweet_id, customer_message, num_turns,
    full_thread_transcript, suggested_intent_regex, sampling_stratum,
    difficulty_flag, gold_intent, intent_ambiguous, gold_escalate,
    escalation_reason, gold_reply_reference, labeler_notes

The last 5 columns are intentionally left BLANK for hand-labeling.
"""

import re
import pandas as pd

RANDOM_SEED = 42

RULES18 = [
    ("Student Discount / Eligibility Verification", r"student|sheerid|\beligib"),
    ("Gift Cards / Redemption Codes",     r"gift card|redeem|promo code|voucher"),
    ("Payment / Card Errors",             r"\bcard (is|was|isn.?t|not) (valid|working|declin)|declin\w*|invalid (card|payment)|card.{0,15}(fail|reject)"),
    ("Account Security / Hacking",        r"hack|hijack|\bsteal\b|unauthoriz|someone (is|has) (using|accessing)"),
    ("Third-party / Platform Integration",r"\broku\b|keynote|control center|smart ?tv|\bxbox\b|playstation|apple watch|car ?play|\balexa\b|google home|\bsonos\b|chromecast|apple tv"),
    ("Ads",                               r"\bads?\b|advert|commercial"),
    ("Billing / Subscription",            r"premium|subscri|billing|\bcharge(d)?\b|payment|paypal|credit card|refund|cancel|\bprice\b|\bcost\b|\$\d|£\d|€\d|\btrial\b|upgrade|downgrade|family plan|family member"),
    ("Login / Account Access",            r"log ?in|log ?out|password|sign ?in|authenticat|facebook connect|account.*(vanish|disab|lock|suspend)|can.?t access"),
    ("Playback Bugs",                     r"shuffle|repeat|\bskip(s|ping)?\b|buffer|glitch|won.?t play|stop(s|ped)? playing|warp speed|random(ly)? play"),
    ("App / Device Technical Issues",     r"crash|freeze|not working|stopped working|\berror\b|\bbug\b|android|iphone|\bios\b|windows|desktop app|mobile app|\bupdate\b|\bversion\b|reinstall|\bslow\b|lag(gy|ging)?|crawl(ing)?"),
    ("Offline / Downloads / Storage",     r"offline|download|\bstorage\b|data usage|\bspace\b|\bgb\b"),
    ("Regional Availability",             r"\bregion\b|\bcountry\b|\babroad\b|\bgeo\b|not available in|launch in"),
    ("Content Availability / Catalog Gaps", r"missing|unavailable|not available|removed|explicit|\balbum\b|\btrack(s)?\b|catalog|bring (back|his|her|their)|where is|duplicate artist|wrong (artist|song)|mislabel"),
    ("Playlist Management",               r"playlist|discover weekly|release radar"),
    ("Feature Requests / Product Feedback", r"\bfeature\b|\bsuggest(ion)?\b|\bwish\b|please add|would be nice|add (a|the) option|\bnotification|\bux\b|user experience"),
    ("Support-Process Complaints",        r"no response|been waiting|days? (now|and) (no|still)|ignored|no one (is helping|responds)|already (dm|emailed)|sitting on.{0,20}(hour|hr)"),
    ("Positive Feedback / Praise",        r"\bthank|\blove\b (you|spotify|this)|amazing|awesome|best app|great job|you.?re the best"),
]
COMPILED = [(label, re.compile(pat, re.IGNORECASE)) for label, pat in RULES18]


def matching_intents(text):
    """Return ALL intents a message matches (not just the first) -- used to
    detect genuine multi-intent ambiguity for the difficulty flag."""
    if not isinstance(text, str) or not text.strip():
        return []
    return [label for label, pat in COMPILED if pat.search(text)]


def classify(text):
    hits = matching_intents(text)
    return hits[0] if hits else "Other / Uncategorised"


def build_thread_transcript(df, conversation_id):
    thread = df[df["conversation_id"] == conversation_id].sort_values("turn_number")
    lines = []
    for _, row in thread.iterrows():
        speaker = "Customer" if row["inbound"] else "SpotifyCares"
        lines.append(f"{speaker}: {row['text']}")
    return "\n".join(lines)


def main():
    df = pd.read_csv("/mnt/user-data/uploads/SpotifyCares.csv")
    inbound = df[df["inbound"] == True].copy()
    first_msgs = inbound[inbound["turn_number"] == 1].copy()

    first_msgs["suggested_intent_regex"] = first_msgs["text"].apply(classify)
    first_msgs["_intent_hits"] = first_msgs["text"].apply(matching_intents)
    first_msgs["_word_count"] = first_msgs["text"].str.split().str.len()

    # thread length per conversation, for difficulty / context richness
    turn_counts = df.groupby("conversation_id")["turn_number"].max()
    first_msgs["num_turns"] = first_msgs["conversation_id"].map(turn_counts)

    def auto_difficulty(row):
        if len(row["_intent_hits"]) >= 2:
            return "hard (multi-intent match)"
        if row["_word_count"] < 6:
            return "hard (very short/low-signal)"
        if row["num_turns"] >= 6:
            return "medium (long thread)"
        return "easy"

    first_msgs["difficulty_flag"] = first_msgs.apply(auto_difficulty, axis=1)

    samples = []

    # 10 per named intent stratum, mixing difficulty where possible
    for intent, _ in RULES18:
        pool = first_msgs[first_msgs["suggested_intent_regex"] == intent]
        if len(pool) == 0:
            continue
        n = min(10, len(pool))
        # prioritize getting at least a couple of "hard" cases per stratum if available,
        # then top up from whichever pool (hard/easy) still has rows left so every
        # stratum reaches its target n whenever the pool is large enough to support it
        hard_pool = pool[pool["difficulty_flag"].str.startswith("hard")]
        easy_pool = pool[~pool["difficulty_flag"].str.startswith("hard")]
        n_hard = min(3, len(hard_pool), n)
        hard_picked = hard_pool.sample(n_hard, random_state=RANDOM_SEED) if n_hard > 0 else hard_pool.iloc[0:0]
        remaining = n - len(hard_picked)
        n_easy = min(remaining, len(easy_pool))
        easy_picked = easy_pool.sample(n_easy, random_state=RANDOM_SEED) if n_easy > 0 else easy_pool.iloc[0:0]
        still_needed = n - len(hard_picked) - len(easy_picked)
        if still_needed > 0:
            leftover_hard = hard_pool.drop(hard_picked.index)
            topup = leftover_hard.sample(min(still_needed, len(leftover_hard)), random_state=RANDOM_SEED) if len(leftover_hard) > 0 else leftover_hard
            picked = pd.concat([hard_picked, easy_picked, topup])
        else:
            picked = pd.concat([hard_picked, easy_picked])
        picked = picked.assign(sampling_stratum=intent)
        samples.append(picked)

    # 20 from Other/Uncategorised -- the long tail needs real representation
    other_pool = first_msgs[first_msgs["suggested_intent_regex"] == "Other / Uncategorised"]
    other_sample = other_pool.sample(min(20, len(other_pool)), random_state=RANDOM_SEED)
    other_sample = other_sample.assign(sampling_stratum="Other / Uncategorised")
    samples.append(other_sample)

    golden = pd.concat(samples).drop_duplicates(subset="tweet_id").reset_index(drop=True)
    golden["example_id"] = [f"EX{idx+1:03d}" for idx in golden.index]
    golden["full_thread_transcript"] = golden["conversation_id"].apply(
        lambda cid: build_thread_transcript(df, cid)
    )

    # Blank judgment columns for hand-labeling
    for col in ["gold_intent", "intent_ambiguous", "gold_escalate",
                "escalation_reason", "gold_reply_reference", "labeler_notes"]:
        golden[col] = ""

    final_cols = [
        "example_id", "conversation_id", "tweet_id", "text", "num_turns",
        "full_thread_transcript", "suggested_intent_regex", "sampling_stratum",
        "difficulty_flag", "gold_intent", "intent_ambiguous", "gold_escalate",
        "escalation_reason", "gold_reply_reference", "labeler_notes",
    ]
    golden = golden[final_cols].rename(columns={"text": "customer_message"})

    print(f"Total draft examples: {len(golden)}")
    print("\nBy stratum:")
    print(golden["sampling_stratum"].value_counts())
    print("\nBy auto difficulty flag:")
    print(golden["difficulty_flag"].value_counts())
